# ASL Sign Language Classification — CNN Experiments
### MSDS 458 — Mary Kate Fitzpatrick
Run each cell in order. Cell 2 will prompt you to upload your two CSV files. Cell 9 prints the full results summary.

In [ ]:
# CELL 1: Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time, warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split

print('TensorFlow:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)
print('Setup complete.')

In [ ]:
# CELL 2: Upload data
# When prompted, upload BOTH sign_mnist_train.csv AND sign_mnist_test.csv
from google.colab import files
print('Please upload sign_mnist_train.csv and sign_mnist_test.csv')
uploaded = files.upload()

train_df = pd.read_csv('sign_mnist_train.csv')
test_df  = pd.read_csv('sign_mnist_test.csv')

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
print(f'Label counts (train):')
print(train_df['label'].value_counts().sort_index())

In [ ]:
# CELL 3: Preprocess
def preprocess(df):
    labels = df['label'].values
    pixels = df.drop('label', axis=1).values
    X = pixels.reshape(-1, 28, 28, 1).astype('float32') / 255.0
    unique_labels = sorted(df['label'].unique())
    label_remap = {old: new for new, old in enumerate(unique_labels)}
    y = np.array([label_remap[l] for l in labels])
    return X, y, unique_labels

X_train_full, y_train_full, unique_labels = preprocess(train_df)
X_test, y_test, _ = preprocess(test_df)

# Letter names matching the dataset labels
letter_names = [chr(65+i) for i in unique_labels]
num_classes = len(unique_labels)

# Train/val split (90/10)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.10, random_state=42, stratify=y_train_full
)

print(f'Number of classes: {num_classes}')
print(f'Letters: {letter_names}')
print(f'Train: {X_train.shape}')
print(f'Val:   {X_val.shape}')
print(f'Test:  {X_test.shape}')
print('Preprocessing complete.')

In [ ]:
# CELL 4: Model builder function
def build_cnn(num_conv_blocks=2, dropout_rate=0.50):
    model = keras.Sequential(name=f'CNN_{num_conv_blocks}blocks_drop{int(dropout_rate*100)}')

    # Conv Block 1
    model.add(layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(2,2))

    # Conv Block 2
    model.add(layers.Conv2D(64, (3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(2,2))

    # Optional Conv Block 3
    if num_conv_blocks == 3:
        model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same'))
        model.add(layers.BatchNormalization())

    # Classifier head
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(num_classes, activation='softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Preview baseline
build_cnn().summary()

In [ ]:
# CELL 5: Train baseline (2 blocks, dropout 0.50)
print('=== BASELINE: 2 conv blocks, dropout 0.50 ===')
tf.random.set_seed(42)
baseline = build_cnn(num_conv_blocks=2, dropout_rate=0.50)
es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

start = time.time()
hist_baseline = baseline.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20, batch_size=64,
    callbacks=[es], verbose=1
)
print(f'Training time: {time.time()-start:.1f}s')

y_pred_baseline = np.argmax(baseline.predict(X_test, verbose=0), axis=1)
print(f'\nBaseline Test Accuracy:  {accuracy_score(y_test, y_pred_baseline):.4f}')
print(f'Baseline Test F1 (macro): {f1_score(y_test, y_pred_baseline, average="macro"):.4f}')

In [ ]:
# CELL 6: Train depth variant (3 conv blocks, dropout 0.50)
print('=== DEPTH VARIANT: 3 conv blocks, dropout 0.50 ===')
tf.random.set_seed(42)
deep_model = build_cnn(num_conv_blocks=3, dropout_rate=0.50)
es2 = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

start = time.time()
hist_deep = deep_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20, batch_size=64,
    callbacks=[es2], verbose=1
)
print(f'Training time: {time.time()-start:.1f}s')

y_pred_deep = np.argmax(deep_model.predict(X_test, verbose=0), axis=1)
print(f'\nDeep Test Accuracy:  {accuracy_score(y_test, y_pred_deep):.4f}')
print(f'Deep Test F1 (macro): {f1_score(y_test, y_pred_deep, average="macro"):.4f}')

In [ ]:
# CELL 7: Dropout variants (0.25 and 0.75, 2 blocks)
results_dropout = {}

for dr in [0.25, 0.75]:
    print(f'=== DROPOUT VARIANT: rate={dr} ===')
    tf.random.set_seed(42)
    m = build_cnn(num_conv_blocks=2, dropout_rate=dr)
    es_d = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    h = m.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=20, batch_size=64,
        callbacks=[es_d], verbose=1
    )
    y_pred = np.argmax(m.predict(X_test, verbose=0), axis=1)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='macro')
    results_dropout[dr] = {'acc': acc, 'f1': f1, 'epochs': len(h.history['loss']), 'preds': y_pred}
    print(f'Dropout {dr} -> Accuracy: {acc:.4f}  F1: {f1:.4f}\n')

# Add baseline and deep to summary
results_dropout[0.50] = {
    'acc': accuracy_score(y_test, y_pred_baseline),
    'f1':  f1_score(y_test, y_pred_baseline, average='macro'),
    'epochs': len(hist_baseline.history['loss']),
    'preds': y_pred_baseline
}

In [ ]:
# CELL 8: Detailed evaluation of best model
# Find best model by accuracy
best_dr = max(results_dropout, key=lambda k: results_dropout[k]['acc'])
best_preds = results_dropout[best_dr]['preds']

# Also check deep model
acc_deep = accuracy_score(y_test, y_pred_deep)
acc_best_dropout = results_dropout[best_dr]['acc']

if acc_deep >= acc_best_dropout:
    final_preds = y_pred_deep
    best_label = '3-block CNN (depth variant)'
else:
    final_preds = best_preds
    best_label = f'2-block CNN dropout={best_dr}'

print(f'Best model: {best_label}')
print(f'\n=== FULL CLASSIFICATION REPORT (best model) ===')
print(classification_report(y_test, final_preds, target_names=letter_names))

print('\n=== CONFUSION MATRIX (best model) ===')
cm = confusion_matrix(y_test, final_preds)
print(cm)

print('\n=== TOP 5 MOST CONFUSED LETTER PAIRS ===')
cm_copy = cm.copy()
np.fill_diagonal(cm_copy, 0)
for _ in range(5):
    idx = np.unravel_index(cm_copy.argmax(), cm_copy.shape)
    print(f'  {letter_names[idx[0]]} mistaken for {letter_names[idx[1]]}: {cm_copy[idx]} times')
    cm_copy[idx] = 0

print('\n=== TRAINING HISTORY (baseline) ===')
for i, (tl, ta, vl, va) in enumerate(zip(
    hist_baseline.history['loss'],
    hist_baseline.history['accuracy'],
    hist_baseline.history['val_loss'],
    hist_baseline.history['val_accuracy']
)):
    print(f'Epoch {i+1:2d}: train_loss={tl:.4f} train_acc={ta:.4f} val_loss={vl:.4f} val_acc={va:.4f}')

In [ ]:
# CELL 9: FINAL SUMMARY — PASTE THIS TO CLAUDE
print('='*65)
print('FINAL RESULTS SUMMARY — PASTE THIS TO CLAUDE')
print('='*65)
print(f'Dataset: {len(train_df)+len(test_df):,} total images')
print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')
print(f'Classes: {num_classes} | Letters: {letter_names}')
print()
print('--- MODEL COMPARISON ---')
print(f'{"Model":<35} {"Accuracy":>10} {"F1 (macro)":>12} {"Epochs":>8}')
print('-'*67)

# Baseline
acc_b = accuracy_score(y_test, y_pred_baseline)
f1_b  = f1_score(y_test, y_pred_baseline, average='macro')
ep_b  = len(hist_baseline.history['loss'])
print(f'{"Baseline (2 blocks, drop=0.50)":<35} {acc_b:>10.4f} {f1_b:>12.4f} {ep_b:>8}')

# Deep
acc_d = accuracy_score(y_test, y_pred_deep)
f1_d  = f1_score(y_test, y_pred_deep, average='macro')
ep_d  = len(hist_deep.history['loss'])
print(f'{"Depth variant (3 blocks, drop=0.50)":<35} {acc_d:>10.4f} {f1_d:>12.4f} {ep_d:>8}')

# Dropout variants
for dr in [0.25, 0.75]:
    r = results_dropout[dr]
    print(f'{f"Dropout variant (2 blocks, drop={dr})":<35} {r["acc"]:>10.4f} {r["f1"]:>12.4f} {r["epochs"]:>8}')

print()
print(f'Best model: {best_label}')
print(f'Best accuracy: {accuracy_score(y_test, final_preds):.4f}')
print(f'Best F1 (macro): {f1_score(y_test, final_preds, average="macro"):.4f}')
print(f'Best precision:  {precision_score(y_test, final_preds, average="macro"):.4f}')
print(f'Best recall:     {recall_score(y_test, final_preds, average="macro"):.4f}')

print()
print('--- TOP 5 CONFUSED LETTER PAIRS (best model) ---')
cm2 = confusion_matrix(y_test, final_preds)
cm_copy2 = cm2.copy()
np.fill_diagonal(cm_copy2, 0)
for _ in range(5):
    idx = np.unravel_index(cm_copy2.argmax(), cm_copy2.shape)
    print(f'  {letter_names[idx[0]]} mistaken for {letter_names[idx[1]]}: {cm_copy2[idx]} times')
    cm_copy2[idx] = 0

print()
print('--- BASELINE TRAINING HISTORY ---')
for i, (tl, ta, vl, va) in enumerate(zip(
    hist_baseline.history['loss'],
    hist_baseline.history['accuracy'],
    hist_baseline.history['val_loss'],
    hist_baseline.history['val_accuracy']
)):
    print(f'Epoch {i+1:2d}: train_loss={tl:.4f} train_acc={ta:.4f} val_loss={vl:.4f} val_acc={va:.4f}')

print('='*65)